# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
**Unit of analysis:** One row represents one content item for one client, aggregated from daily search-performance observations.

**Time window:** I will use February 2026 as the feature window and March 2026 as the outcome window. Features should only use information available by the decision point, while the later outcome is used only as the label/proxy.

In [17]:
import os
import duckdb
from google.colab import userdata

# Load the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("HF_TOKEN loaded successfully")
print("Warehouse connection ready")
print("Feature window: February 2026")
print("Label window: March 2026")

HF_TOKEN loaded successfully
Warehouse connection ready
Feature window: February 2026
Label window: March 2026


## 2. Fields: feature / label / context / excluded

**Features:** impressions, clicks, average position, CTR, and content age. These are observable signals that can be available before the decision point.

**Label:** March 2026 trend/outcome used as the proxy for whether a page showed a subsequent decline.

**Context:** client identifier and content identifier are used to define the grain and group observations, but they are not model features.

**Excluded:** any field that is derived from the future outcome or from a decision already made, because it would leak the answer into the model. I will also exclude the final-month `_sample` data from feature development because it is the sealed outcome month.

In [18]:

print(con.execute("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
""").df().to_string(index=False))

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
**Five features**

1. `avg_gsc_impressions` — knowable at the decision moment because it is observed during the February 2026 feature window.
2. `avg_gsc_clicks` — knowable at the decision moment because it is observed during the February 2026 feature window.
3. `avg_gsc_avg_position` — knowable at the decision moment because it is observed during the February 2026 feature window.
4. `avg_ga4_sessions` — knowable at the decision moment because it is observed during the February 2026 feature window.
5. `avg_scroll_events` — knowable at the decision moment because it is observed during the February 2026 feature window.

**Deliberate leakage experiment**

I intentionally added `march_impressions` from the March 2026 outcome window to the February feature frame. This is leakage because March information would not be available at the February decision moment. A real model could therefore appear stronger than it actually is. I will remove this field and keep only features that were knowable during the feature window.
**Leakage result**

When I deliberately added `march_impressions`, a future March 2026 field, the quick ranking reached Precision@50 = 1.000. This looks perfect, but it is not a valid model result because March information was not available at the February decision moment. The experiment shows how leakage can make a model appear much stronger than it really is.

In [19]:
grain_check = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || report_date)
        AS distinct_client_content_day
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,distinct_client_content_day
0,9841378,9841378


In [20]:
count_dates = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(count_dates)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [21]:
availability = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [22]:
feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_avg_position) AS avg_gsc_position,
    AVG(ga4_sessions) AS avg_ga4_sessions,
    AVG(scroll_events) AS avg_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

feature_df = con.execute(feature_query).df()

print("Feature frame shape:", feature_df.shape)
display(feature_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (321546, 7)


,client_hash_id,content_hash_id,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_position,avg_ga4_sessions,avg_scroll_events
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,0.000000,12.946228,0.000000,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,0.214286,6.495085,0.214286,0.000000
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,0.000000,10.490023,0.035714,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,0.107143,38.436254,0.214286,0.035714
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,0.071429,9.710810,0.107143,0.035714
5,client_e547b89c05043229,content_a2bd730a7cf68316,19.678571,0.035714,6.017373,0.107143,0.000000
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,10.392857,0.000000,4.333115,0.107143,0.000000
7,client_e547b89c05043229,content_babd931911c9ee33,95.714286,1.107143,5.046562,0.857143,0.178571
8,client_e547b89c05043229,content_9c36ace83c73b5eb,14.857143,0.035714,40.800604,0.035714,0.000000
9,client_e547b89c05043229,content_431784c057b25a5d,130.035714,0.214286,8.784133,0.642857,0.000000


In [23]:
label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_query).df()

print("Label frame shape:", label_df.shape)
display(label_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label frame shape: (331437, 3)


,client_hash_id,content_hash_id,march_impressions
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0
5,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0
6,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0
7,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,134.0
8,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0
9,client_62f4a7e64f5e0096,content_225dc9235023be5f,488.0


In [24]:
leaky_df = feature_df.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Leaky frame shape:", leaky_df.shape)

# Deliberate leakage:
# March impressions are from the future outcome window.
print("March outcome has been intentionally added as a leaky feature.")
display(leaky_df.head(10))

Leaky frame shape: (303572, 8)
March outcome has been intentionally added as a leaky feature.


,client_hash_id,content_hash_id,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_position,avg_ga4_sessions,avg_scroll_events,march_impressions
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,0.000000,12.946228,0.000000,0.000000,315.0
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,0.214286,6.495085,0.214286,0.000000,14536.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,0.000000,10.490023,0.035714,0.000000,387.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,0.107143,38.436254,0.214286,0.035714,4697.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,0.071429,9.710810,0.107143,0.035714,1004.0
5,client_e547b89c05043229,content_a2bd730a7cf68316,19.678571,0.035714,6.017373,0.107143,0.000000,551.0
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,10.392857,0.000000,4.333115,0.107143,0.000000,344.0
7,client_e547b89c05043229,content_babd931911c9ee33,95.714286,1.107143,5.046562,0.857143,0.178571,4946.0
8,client_e547b89c05043229,content_9c36ace83c73b5eb,14.857143,0.035714,40.800604,0.035714,0.000000,302.0
9,client_e547b89c05043229,content_431784c057b25a5d,130.035714,0.214286,8.784133,0.642857,0.000000,1871.0


In [25]:
# Create a simple March outcome label for the leakage demonstration.
# This is only for the deliberate leakage experiment.
leaky_df["march_declined_proxy"] = (
    leaky_df["march_impressions"] == 0
).astype(int)

print(
    "Rows with March outcome proxy = 1:",
    leaky_df["march_declined_proxy"].sum()
)
print(
    "Outcome proxy rate:",
    round(leaky_df["march_declined_proxy"].mean(), 3)
)

Rows with March outcome proxy = 1: 142033
Outcome proxy rate: 0.468


In [26]:
# Deliberately use the future March feature to rank pages.
scores = leaky_df["march_impressions"].fillna(0).abs()

# Lower March impressions = higher refresh concern in this toy proxy.
leaky_score = -scores

order = leaky_score.to_numpy().argsort()[::-1]
top50 = leaky_df["march_declined_proxy"].to_numpy()[order[:50]]

print("Leaky Precision@50:", round(top50.mean(), 3))

Leaky Precision@50: 1.0


In [27]:
# Remove the deliberately leaked future feature.
honest_features = [
    "avg_gsc_impressions",
    "avg_gsc_clicks",
    "avg_gsc_position",
    "avg_ga4_sessions",
    "avg_scroll_events"
]

honest_feature_df = leaky_df[
    ["client_hash_id", "content_hash_id"] + honest_features
].copy()

print("Honest feature frame shape:", honest_feature_df.shape)
print("Leaked feature removed:", "march_impressions" not in honest_feature_df.columns)
display(honest_feature_df.head(10))

Honest feature frame shape: (303572, 7)
Leaked feature removed: True


,client_hash_id,content_hash_id,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_position,avg_ga4_sessions,avg_scroll_events
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,0.000000,12.946228,0.000000,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,0.214286,6.495085,0.214286,0.000000
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,0.000000,10.490023,0.035714,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,0.107143,38.436254,0.214286,0.035714
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,0.071429,9.710810,0.107143,0.035714
5,client_e547b89c05043229,content_a2bd730a7cf68316,19.678571,0.035714,6.017373,0.107143,0.000000
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,10.392857,0.000000,4.333115,0.107143,0.000000
7,client_e547b89c05043229,content_babd931911c9ee33,95.714286,1.107143,5.046562,0.857143,0.178571
8,client_e547b89c05043229,content_9c36ace83c73b5eb,14.857143,0.035714,40.800604,0.035714,0.000000
9,client_e547b89c05043229,content_431784c057b25a5d,130.035714,0.214286,8.784133,0.642857,0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
**Data limits**

This slice has uneven data availability across clients and content items. The March 2026 results are also an outcome window, so they cannot be used as features for a February decision. Missing GSC or GA4 data means some signals are unavailable for some rows. The data can support measured and directional decision-support, but it cannot establish that a content action caused an improvement or decline.
**Data limits**

The warehouse slice has uneven availability across signals. In the honest feature frame of 303,572 client-content rows, `avg_gsc_avg_position` is missing for 158,293 rows, while `avg_ga4_sessions` and `avg_scroll_events` are each missing for 131,711 rows. This means some pages do not have all signals available at the decision moment. The data can support measured and directional decision-support, but it cannot establish that a content action caused an improvement or decline.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows in honest feature frame:", len(honest_feature_df))
print("Missing values by feature:")
display(honest_feature_df[honest_features].isna().sum().to_frame("missing_rows"))

Rows in honest feature frame: 303572
Missing values by feature:


,missing_rows
avg_gsc_impressions,0
avg_gsc_clicks,0
avg_gsc_position,158293
avg_ga4_sessions,131711
avg_scroll_events,131711


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.